<a href="https://colab.research.google.com/github/Pavan186186/DNA/blob/main/DNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the bioinformatics toolkit
!pip install -q biopython

import os
from Bio import Entrez, SeqIO
import pandas as pd

# HIGH STANDARD: NCBI requires an email to contact you if your script
# causes server strain. This is a requirement for "professional" use.
Entrez.email = "your_email@example.com"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.5 MB/s eta 0:00:00


In [3]:
import time
from Bio import Entrez, SeqIO
import pandas as pd

def fetch_reptile_dataset_resilient(taxon_name, max_records=3):
    """
    An improved, resilient scraper with retry logic and targeted field tags.
    """
    print(f"--- Initiating Resilient Search for: {taxon_name} ---")

    # 1. Refined Search Query using Field Tags [ORGN] and [TITL]
    # We target 'RefSeq' records which are the gold standard for accuracy.
    search_query = f"{taxon_name}[Organism] AND complete genome[Title] AND mitochondrion[Filter] AND srcdb_refseq[Properties]"

    try:
        # Search for IDs
        search_handle = Entrez.esearch(db="nucleotide", term=search_query, retmax=max_records)
        search_results = Entrez.read(search_handle)
        search_handle.close()

        id_list = search_results.get("IdList", [])
        if not id_list:
            print(f"No RefSeq records found for {taxon_name}. Trying broader search...")
            # Fallback to a simpler search if RefSeq is too restrictive
            search_query_simple = f"{taxon_name}[Organism] AND complete genome[Title] AND mitochondrion[Filter]"
            search_handle = Entrez.esearch(db="nucleotide", term=search_query_simple, retmax=max_records)
            search_results = Entrez.read(search_handle)
            id_list = search_results.get("IdList", [])

        print(f"Validated IDs found: {id_list}")

        dataset = []
        for ncbi_id in id_list:
            # High Standard: Rate limiting to avoid 429 Errors
            time.sleep(1)

            print(f"Fetching Record: {ncbi_id}...")
            # Fetch in FASTA format (text-based) to avoid XML parsing errors
            fetch_handle = Entrez.efetch(db="nucleotide", id=ncbi_id, rettype="fasta", retmode="text")
            raw_data = fetch_handle.read()
            fetch_handle.close()

            # Use a StringIO to parse the text as a sequence record
            from io import StringIO
            record = SeqIO.read(StringIO(raw_data), "fasta")

            dataset.append({
                "species_name": " ".join(record.description.split()[:2]), # Cleaner species naming
                "ncbi_accession": record.id,
                "sequence_length": len(record.seq),
                "dna_sequence": str(record.seq)
            })

        return pd.DataFrame(dataset)

    except Exception as e:
        print(f"Critical Failure on {taxon_name}: {e}")
        return pd.DataFrame()

# Execute with the new logic
df_crocs = fetch_reptile_dataset_resilient("Crocodylia", max_records=3)
df_lizards = fetch_reptile_dataset_resilient("Squamata", max_records=3)

# Combine and Preview
if not df_crocs.empty or not df_lizards.empty:
    master_df = pd.concat([df_crocs, df_lizards], ignore_index=True)
    print("\n--- MASTER DATABASE CREATED ---")
    print(master_df[["species_name", "sequence_length"]])
else:
    print("Database is empty. Check internet connection or NCBI API status.")

--- Initiating Resilient Search for: Crocodylia ---
Validated IDs found: ['669017071', '336055628', '336055569']
Fetching Record: 669017071...
Fetching Record: 336055628...
Fetching Record: 336055569...
--- Initiating Resilient Search for: Squamata ---
Critical Failure on Squamata: Search Backend failed: 		


--- MASTER DATABASE CREATED ---
             species_name  sequence_length
0  NC_024513.1 Crocodylus            16795
1  NC_015651.1 Crocodylus            16894
2  NC_015648.1 Crocodylus            16828


In [4]:
!pip install -q together

import os
import together
import numpy as np

# Replace with your actual key
os.environ["TOGETHER_API_KEY"] = "key_CZWCB5nDX9E1zAacuYoVZ"
together.api_key = os.environ["TOGETHER_API_KEY"]

print("Phase 2 Environment Ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 14.4 MB/s eta 0:00:00
Phase 2 Environment Ready.


In [5]:
def get_dna_embeddings(dna_sequence):
    """
    Sends DNA to Evo-1 to get the 'hidden state' or embedding.
    We use the first 1,000 base pairs as a representative 'signature'.
    """
    # Clean the sequence (ensure uppercase)
    clean_seq = dna_sequence.upper()[:1000]

    print(f"Generating Embeddings for sequence (Length: {len(clean_seq)})...")

    try:
        # We use Evo-1-131k-base
        extract = together.Complete.create(
            model="togethercomputer/evo-1-131k-base",
            prompt=clean_seq,
            max_tokens=1, # We only want the internal representation, not a completion
            logprobs=1,
            echo=True
        )

        # In a full production environment, we would extract the hidden layers.
        # For this Colab MVP, we will use the token log-probabilities as a
        # unique 'Genomic Signature' vector.

        token_logprobs = extract['output']['choices'][0]['logprobs']['token_logprobs']
        # Handle potential None values and convert to a fixed-size vector
        vector = np.array([lp if lp is not None else -10.0 for lp in token_logprobs])

        return vector

    except Exception as e:
        print(f"Embedding Error: {e}")
        return None

# Apply the embedding to our Crocodilian database
print("--- Processing Master Database ---")
master_df['embeddings'] = master_df['dna_sequence'].apply(get_dna_embeddings)

# Remove any failures
master_df = master_df.dropna(subset=['embeddings'])
print(f"Successfully embedded {len(master_df)} species.")

--- Processing Master Database ---
Generating Embeddings for sequence (Length: 1000)...
Embedding Error: module 'together' has no attribute 'Complete'
Generating Embeddings for sequence (Length: 1000)...
Embedding Error: module 'together' has no attribute 'Complete'
Generating Embeddings for sequence (Length: 1000)...
Embedding Error: module 'together' has no attribute 'Complete'
Successfully embedded 0 species.


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate similarity matrix
if len(master_df) > 1:
    matrix = np.stack(master_df['embeddings'].values)
    similarity = cosine_similarity(matrix)

    plt.figure(figsize=(8, 6))
    sns.heatmap(similarity, annot=True, xticklabels=master_df['species_name'], yticklabels=master_df['species_name'], cmap='viridis')
    plt.title("Genomic Similarity Matrix (Evo-1 Vectors)")
    plt.show()
else:
    print("Not enough data to compare. Ensure Phase 1 and 2 ran correctly.")

Not enough data to compare. Ensure Phase 1 and 2 ran correctly.
